In [ ]:
import json
import os

from dotenv import load_dotenv
from azure.cosmos import CosmosClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
import requests
from pathlib import Path

from Backend.models import Product, ProductList

loaded = load_dotenv()
if not loaded:
    raise RuntimeError(".env file could not be loaded.")

key_vault_url = os.environ["KEY_VAULT_URL"]
secret_name = os.environ["COSMOS_DB_CONNECTION_STRING_SECRET_NAME"]

credential = DefaultAzureCredential()
secret_client = SecretClient(vault_url=key_vault_url, credential=credential)
connection_string = secret_client.get_secret(secret_name).value

if connection_string is None:
    raise RuntimeError(f"Key Vault secret '{secret_name}' has no value.")

client = CosmosClient.from_connection_string(connection_string)

DatabaseProxy = client.create_database_if_not_exists("test_db_5")

ContainerProxy = DatabaseProxy.create_container_if_not_exists(
    id="product",
    partition_key={"paths": ["/categoryId"], "kind": "Hash"}
)

# Create a document
# ContainerProxy.upsert_item({"id": "1", "categoryId": "electronics"})

# items = ContainerProxy.query_items(query="SELECT * FROM product", enable_cross_partition_query=True)

# Add product data to database using batch operations
# Get cosmic works product data from github
# product_raw_data = "https://cosmosdbcosmicworks.blob.core.windows.net/cosmic-works-small/product.json"
# product_data = ProductList(items=[Product(**data) for data in requests.get(product_raw_data).json()])
# for prod in product_data.items:
#     print(prod)
#     ContainerProxy.upsert_item(prod.model_dump(by_alias=True))

# product_raw_data = "https://cosmosdbcosmicworks.blob.core.windows.net/cosmic-works-small/product.json"
# product_data = ProductList(items=[Product(**data) for data in requests.get(product_raw_data).json()])
# for prod in product_data.items:
#     ContainerProxy.upsert_item(prod.model_dump(by_alias=True))

product_file = Path("products.json")
with product_file.open(encoding="utf-8") as file:
    product_data = ProductList(
        items=[Product(**data) for data in json.load(file)]
    )

for prod in product_data.items:
    ContainerProxy.upsert_item(prod.model_dump(by_alias=True))